In [1]:
import sys
import os
sys.path.append(os.path.abspath('../../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ucphm import *
from functions.new_variables import *
from functions.ensembles import *
from scipy.stats import skew, shapiro

In [2]:
# Configure pandas display options to show all columns and rows
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [3]:
dataset = pd.read_excel('../../../BreastCareCond.xlsx')
data_code = pd.read_excel('../../../Codificação variaveis.xlsx')

In [4]:
#Defining the proposed has good treatment variables to use (the empty ones were removed)
relevant_variables = ['PERSON_ID', 'EBC', 'Survival days', 'Regional LN recurrence', 'Date of 1st regional recurrence']
numerical_variables = ['Age', 'Ki67 positive cells (%) at CB']
categorical_variables = ['Grade at CB', 'Side location of the lesion',
                         'Type of invasion at CB', 'Isotype at CB', 
                         'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 
                         'Her2 overexpression (with immunohystochemistry) at CB', 
                         'N (Regional nodes affected)']
                         
date_columns = ['Date of CB ', 'Date of 1st breast operation', 'Date of intervention on sentinel lymph node',
                'Date of intervention (axilla)']

In [5]:
#correcting th error in the column name
dataset.rename(columns={'Istotype at CB': 'Isotype at CB'}, inplace=True)

In [6]:
dataset = date_of_diagnosis(dataset, date_columns)

relevant_dates = ['Date of diagnosis', 'Date of 1st regional recurrence']
dataset = remove_zeros_na(dataset, relevant_dates)
dataset = format_date(dataset, relevant_dates)

dataset = survival_days(dataset, ['Date of diagnosis'], ['Date of 1st regional recurrence'])
dataset = tte_negative_patients(dataset)
dataset = filter_EBC(dataset)
dataset.reset_index(drop=True, inplace=True)

In [7]:
# pT(dataset)
# cT (dataset)
# clinical_staging(dataset)
# didn't place the ct in the new variables, the pt is already at the top
# new_variables = ['Cancer stage']
# categorical_variables = categorical_variables + new_variables

In [8]:
change_to_numeric(dataset, numerical_variables)
change_to_categorical(dataset, categorical_variables)

Done
Done


In [9]:
diagnostic_panel_fields = [
    'Ki67 positive cells (%) at CB',
    'Grade at CB',
    'Type of invasion at CB',
    'Isotype at CB',
    'Oestrogen receptor status at CB',
    'Progesterone receptor status at CB',
    'Her2 overexpression (with immunohystochemistry) at CB',
]

panel_missing = dataset[diagnostic_panel_fields].isnull()
n_panel_missing = panel_missing.sum(axis=1)

# The 9.6% pattern identified in the missingness analysis: PR, HER2 and Ki67
# missing together (exactly 3 of the 7 fields missing).
three_missing_pattern = (
    panel_missing['Progesterone receptor status at CB']
    & panel_missing['Her2 overexpression (with immunohystochemistry) at CB']
    & panel_missing['Ki67 positive cells (%) at CB']
    & (n_panel_missing == 3)
)

# Only the two systematic patterns found in the analysis get their own flag.
# Any other combination of missing fields is left to be captured by each
# field's own missing-indicator one-hot columns further down (all-zero across
# that field's dummies when missing) -- no separate "other" bucket here.
dataset['Diagnostic panel missingness pattern'] = 'complete'
dataset.loc[three_missing_pattern, 'Diagnostic panel missingness pattern'] = 'missing_3'
dataset.loc[n_panel_missing == 7, 'Diagnostic panel missingness pattern'] = 'missing_7'

dataset['Diagnostic panel missingness pattern'].value_counts()

Diagnostic panel missingness pattern
complete     2795
missing_3     305
missing_7     131
Name: count, dtype: int64

In [10]:
columns_to_map = {
    'Side location of the lesion': {'L': 0, 'R': 1},
    'Oestrogen receptor status at CB': {1: 0, 2: 1},
    'Type of invasion at CB': {1: 0, 2: 1},
    'Progesterone receptor status at CB': {1: 0, 2: 1},
    'N (Regional nodes affected)': {0: 0, 1: 1, 2: 2_3, 3: 2_3}
    }

for column, mapping in columns_to_map.items():
    dataset[column] = dataset[column].map(mapping)

In [11]:
# Side location and N (Regional nodes affected) are mode-filled unconditionally
# here. The 6 diagnostic-panel categorical fields (Grade, Type of invasion,
# Isotype, ER, PR, HER2) are NOT mode-filled yet: they keep their missing
# values for now so the systematic-pattern check above stayed based on raw
# missingness. They get mode-filled selectively (only where missingness isn't
# part of the missing_3/missing_7 pattern) a couple of cells down.
mode_variables = ['Side location of the lesion', 'N (Regional nodes affected)']
fill_with_mode(dataset, mode_variables)

'Done'

In [12]:
median = ['Ki67 positive cells (%) at CB']
median_imputer(dataset, numerical_variables)

'Done'

In [13]:
# Missingness that isn't part of the two systematic patterns identified above
# (missing_3, missing_7) is incidental rather than systematic, so it's
# imputed with the field's mode -- otherwise it would silently collapse into
# the reference/dropped category during the encoding step below with no
# signal anywhere that it was ever missing. Missingness that *is* part of
# missing_3 / missing_7 is left as NaN: it's still tracked by the
# 'Diagnostic panel missingness pattern' flag, so mode-imputing it here would
# just throw that signal away.
diagnostic_categorical_fields = ['Grade at CB', 'Type of invasion at CB', 'Isotype at CB',
                                 'Oestrogen receptor status at CB', 'Progesterone receptor status at CB',
                                 'Her2 overexpression (with immunohystochemistry) at CB']

pattern_covered = dataset['Diagnostic panel missingness pattern'] != 'complete'

for field in diagnostic_categorical_fields:
    orphaned_missing = dataset[field].isnull() & ~pattern_covered
    mode_value = dataset[field].mode(dropna=True).iloc[0]
    dataset.loc[orphaned_missing, field] = mode_value

In [14]:
# removi o'cT', 'Cancer stage', e o Classification with respect to other lesions e o Disease extent
# By this point, the only missing values left in the 6 diagnostic-panel
# fields belong to the missing_3 / missing_7 patients (orphaned/incidental
# missingness was already mode-imputed above). Grade, Isotype and HER2 have
# 3+ categories and still get one-hot encoded, now with drop_first so the
# reference category is dropped (missing_3/missing_7 patients end up all-zero
# across the remaining dummy columns too, same as the dropped reference
# category -- but they're still distinguishable via the pattern flag below).
# Type of invasion, ER and PR are already binary (0/1) after the earlier
# mapping step, so one-hot encoding them would be redundant -- they're kept
# as a single column, with missing values filled to 0 (the reference value),
# matching the drop_first treatment used for the multi-category fields above.
# N and the pattern flag (missing_3 / missing_7) have no missing values left,
# so they still use plain drop-first one-hot encoding.
diagnostic_multi_category_variables = ['Grade at CB', 'Isotype at CB',
                                        'Her2 overexpression (with immunohystochemistry) at CB']
diagnostic_binary_variables = ['Type of invasion at CB', 'Oestrogen receptor status at CB',
                               'Progesterone receptor status at CB']
diagnostic_missing_variables = diagnostic_multi_category_variables + diagnostic_binary_variables

dataset, diagnostic_new_columns = one_hot_encode_with_missing_indicator(
    dataset, diagnostic_multi_category_variables, drop_first=True
)

for variable in diagnostic_binary_variables:
    dataset[variable] = dataset[variable].fillna(0)
diagnostic_new_columns = diagnostic_new_columns + diagnostic_binary_variables

one_hot_variables = ['N (Regional nodes affected)', 'Diagnostic panel missingness pattern']
ohe_dataset, new_column_names = one_hot_encoding(dataset, one_hot_variables, drop_first='first')

In [15]:
categorical_variables = [col for col in categorical_variables if col not in diagnostic_missing_variables and col not in one_hot_variables]
categorical_variables.extend(diagnostic_new_columns)
categorical_variables.extend(new_column_names.tolist())

selected_variables = relevant_variables + numerical_variables + categorical_variables

In [16]:
treatment_variables = ['Radiotherapy (RT) performed', 'Conventional RT fraction', 'Total administered dose', 'Boost dose administered', 
                       'Radiotherapy on chest wall', 'Radiotherapy on supraclavicular area', 'Radiotherapy on internal mammary chain',
                         'State (if hormone therapy prescribed)', 'Treatment in association with chemotherapy', 
                        'Chemotherapy performed', 'Adjuvant chemotherapy', 'Adjuvant therapy with biological drugs', 'Endocrine therapy performed']
                        # 'Date of first administration of biological drugs for adjuvant therapy', 'Final recommendations to patient']
selected_variables_treat = selected_variables + treatment_variables

survival_dataset = ohe_dataset[selected_variables_treat].copy()

Now the treatment variables, just because it was already made

In [17]:
dataset = survival_dataset

In [18]:
numerical_data, categorical_data = type_separation(data_code, treatment_variables)
len(numerical_data), len(categorical_data)

(3, 10)

In [19]:
replace_missing_values(dataset, numerical_data, 0)

Done


In [20]:
# dataset.drop(index=0, inplace=True) removed the drop of the patient
dataset.reset_index(inplace=True, drop=True)
replace_missing_values(dataset, categorical_data, 0)

Done


In [21]:
variable_two_to_zero = ['Adjuvant chemotherapy', 'Chemotherapy performed', 'Endocrine therapy performed']
two_to_zero(dataset, variable_two_to_zero)

bio_variable = 'Adjuvant therapy with biological drugs'
dataset = encode_bio_therapy(dataset, bio_variable)

In [22]:
feature_columns = ['Adjuvant therapy using biological drugs'] #, 'Trastuzumab or other drugs']
feature_columns = feature_columns + categorical_data + numerical_data

treatment_variables = [col for col in feature_columns if col not in bio_variable]

In [23]:
# Create a derived neojuven therapy only variable.
# It is 1 when adjuvant therapy with biological drugs is 0 and endocrine therapy performed is 1.

dataset['Neoadjuvant (Only) therapy with biological drugs'] = (
    (dataset['Adjuvant therapy using biological drugs'] == 0) &
    (dataset['Endocrine therapy performed'] == 1)
).astype(int)

treatment_variables = [
    col for col in treatment_variables
    if col != 'Endocrine therapy performed'
]
treatment_variables.append('Neoadjuvant (Only) therapy with biological drugs')

In [24]:
to_remove = ['PERSON_ID', 'Regional LN recurrence', 'Date of 1st regional recurrence', 'EBC', 'Survival days']
important_variables = ['PERSON_ID', 'Regional LN recurrence', 'Survival days']
diagnosis_variables = [col for col in selected_variables if col not in to_remove]
selected_columns = important_variables + treatment_variables + diagnosis_variables

In [25]:
survival_dataset = dataset[selected_columns]

In [26]:
# Find duplicated patients
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False) & (survival_dataset['Regional LN recurrence'] == 1))

# For each duplicated PERSON_ID with recurrence, keep only the first row
to_remove = (
    survival_dataset[mask]
    .groupby('PERSON_ID')
    .tail(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [27]:
# Check if there are any NaN values in the dataset
contains_nan = survival_dataset.isnull().any().any()
print("Does survival_dataset contain NaN values?", contains_nan)

# Optionally, to see which columns contain NaN values
nan_columns = survival_dataset.columns[survival_dataset.isnull().any()].tolist()
print("Columns with NaN values:", nan_columns)

Does survival_dataset contain NaN values? False
Columns with NaN values: []


In [28]:
survival_dataset.to_csv("data.csv")

In [29]:
treatment_variables.remove('Radiotherapy (RT) performed')

In [30]:
with open("diagnosis.txt", "w") as file:
    for item in diagnosis_variables:
        file.write(str(item) + "\n")
with open("treatment.txt", "w") as file:
    for item in treatment_variables:
        file.write(str(item) + "\n")